# Understanding `torch.einsum`

This notebook is a teaching-first guide to Einstein summation in PyTorch.

It starts with the simplest cases and builds toward patterns you see in real models.
By the end you should be able to read and write equations that use:

- letters such as `i`, `j`, `k`, `b`, `d`
- commas `,` to separate operands
- arrows `->` to control the output
- ellipsis `...` to hide arbitrary batch dimensions
- repeated labels for diagonals and contractions

The examples are intentionally small so you can inspect actual tensor values, not only shapes.


In [34]:
import torch

torch.set_printoptions(linewidth=120, sci_mode=False, precision=3)


def describe(name, equation, *operands):
    result = torch.einsum(equation, *operands)
    print(f"{name}")
    print(f"equation: {equation}")
    for idx, operand in enumerate(operands, start=1):
        print(f"operand {idx}: shape={tuple(operand.shape)}")
        print(operand)
    print(f"result: shape={tuple(result.shape)}")
    print(result)
    print("-" * 100)
    return result


## Mental Model

A good way to read `einsum` is:

1. each letter names one axis
2. operands are separated by commas
3. labels repeated across operands are aligned and multiplied together
4. labels that disappear from the output are summed over
5. labels repeated inside one operand mean "take the diagonal" across those axes
6. `->` makes the output explicit and fixes the axis order
7. `...` stands for any number of extra dimensions

A practical rule: if the equation is not obvious at a glance, write a legend for the letters before you write the code.


In [35]:
# Small, deterministic tensors used across the notebook
v = torch.tensor([2.0, 3.0, 5.0])
w = torch.tensor([7.0, 11.0, 13.0])

A = torch.arange(1.0, 10.0).reshape(3, 3)
B = torch.arange(1.0, 13.0).reshape(3, 4)
B2 = torch.arange(21.0, 33.0).reshape(3, 4)
C = torch.arange(1.0, 9.0).reshape(4, 2)

print('v shape:', v.shape)
print(v)
print('w shape:', w.shape)
print(w)
print('A shape:', A.shape)
print(A)
print('B shape:', B.shape)
print(B)
print('B2 shape:', B2.shape)
print(B2)
print('C shape:', C.shape)
print(C)


v shape: torch.Size([3])
tensor([2., 3., 5.])
w shape: torch.Size([3])
tensor([ 7., 11., 13.])
A shape: torch.Size([3, 3])
tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])
B shape: torch.Size([3, 4])
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.]])
B2 shape: torch.Size([3, 4])
tensor([[21., 22., 23., 24.],
        [25., 26., 27., 28.],
        [29., 30., 31., 32.]])
C shape: torch.Size([4, 2])
tensor([[1., 2.],
        [3., 4.],
        [5., 6.],
        [7., 8.]])


## 1. One Tensor, No Commas Yet

Start with one operand.
This is the easiest place to understand what the labels mean.

- keeping a label keeps that axis
- reordering labels reorders axes
- removing a label sums over that axis


In [36]:
describe('Identity', 'i->i', v)
describe('Transpose a matrix', 'ij->ji', B)
describe('Sum rows (reduce columns)', 'ij->i', B)
describe('Sum columns (reduce rows)', 'ij->j', B)
describe('Sum every element', 'ij->', B)


Identity
equation: i->i
operand 1: shape=(3,)
tensor([2., 3., 5.])
result: shape=(3,)
tensor([2., 3., 5.])
----------------------------------------------------------------------------------------------------
Transpose a matrix
equation: ij->ji
operand 1: shape=(3, 4)
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.]])
result: shape=(4, 3)
tensor([[ 1.,  5.,  9.],
        [ 2.,  6., 10.],
        [ 3.,  7., 11.],
        [ 4.,  8., 12.]])
----------------------------------------------------------------------------------------------------
Sum rows (reduce columns)
equation: ij->i
operand 1: shape=(3, 4)
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.]])
result: shape=(3,)
tensor([10., 26., 42.])
----------------------------------------------------------------------------------------------------
Sum columns (reduce rows)
equation: ij->j
operand 1: shape=(3, 4)
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7

tensor(78.)

## 2. Repeated Labels Inside One Operand: Diagonal and Trace

When the same label appears twice in **one operand**, `einsum` matches those two axes and keeps only the diagonal.

For a square matrix `A`:

- `ii->i` extracts the diagonal
- `ii->` extracts the diagonal and then sums it, which is the trace

If you want the product of the diagonal entries, `einsum` gives you the diagonal and then you can apply `.prod()`.


In [37]:
diagonal = describe('Diagonal extraction', 'ii->i', A)
trace = describe('Trace', 'ii->', A)
diagonal_product = torch.einsum('ii->i', A).prod()

print('Diagonal product = product of the diagonal entries')
print(diagonal_product)


Diagonal extraction
equation: ii->i
operand 1: shape=(3, 3)
tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])
result: shape=(3,)
tensor([1., 5., 9.])
----------------------------------------------------------------------------------------------------
Trace
equation: ii->
operand 1: shape=(3, 3)
tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])
result: shape=()
tensor(15.)
----------------------------------------------------------------------------------------------------
Diagonal product = product of the diagonal entries
tensor(45.)


## 3. Commas Mean Multiple Operands

A comma separates different tensors.

Example: `i,i->`

- the first `i` comes from the first vector
- the second `i` comes from the second vector
- matching positions are multiplied
- `i` does not appear on the right side, so PyTorch sums over it

That is exactly a dot product.


In [38]:
describe('Dot product', 'i,i->', v, w)
describe('Outer product', 'i,j->ij', v, w)
describe('Elementwise / Hadamard product', 'ij,ij->ij', B, B2)
describe('Frobenius inner product of two matrices', 'ij,ij->', B, B2)


Dot product
equation: i,i->
operand 1: shape=(3,)
tensor([2., 3., 5.])
operand 2: shape=(3,)
tensor([ 7., 11., 13.])
result: shape=()
tensor(112.)
----------------------------------------------------------------------------------------------------
Outer product
equation: i,j->ij
operand 1: shape=(3,)
tensor([2., 3., 5.])
operand 2: shape=(3,)
tensor([ 7., 11., 13.])
result: shape=(3, 3)
tensor([[14., 22., 26.],
        [21., 33., 39.],
        [35., 55., 65.]])
----------------------------------------------------------------------------------------------------
Elementwise / Hadamard product
equation: ij,ij->ij
operand 1: shape=(3, 4)
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.]])
operand 2: shape=(3, 4)
tensor([[21., 22., 23., 24.],
        [25., 26., 27., 28.],
        [29., 30., 31., 32.]])
result: shape=(3, 4)
tensor([[ 21.,  44.,  69.,  96.],
        [125., 156., 189., 224.],
        [261., 300., 341., 384.]])
----------------------------

tensor(2210.)

## 4. Classic Linear Algebra Patterns

A large fraction of day-to-day `einsum` usage is just familiar linear algebra written in index form.

This is where `einsum` becomes useful: the notation makes the contraction dimensions explicit.


In [39]:
vector4 = torch.tensor([1.0, 2.0, 3.0, 4.0])
vector3 = torch.tensor([1.0, 2.0, 3.0])

X = torch.arange(1.0, 25.0).reshape(2, 3, 4)
Y = torch.arange(1.0, 41.0).reshape(2, 4, 5)

describe('Matrix-vector product', 'ij,j->i', B, vector4)
describe('Vector-matrix product', 'i,ij->j', vector3, B)
describe('Matrix-matrix product', 'ik,kj->ij', B, C)
describe('Batched matrix multiplication', 'bij,bjk->bik', X, Y)


Matrix-vector product
equation: ij,j->i
operand 1: shape=(3, 4)
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.]])
operand 2: shape=(4,)
tensor([1., 2., 3., 4.])
result: shape=(3,)
tensor([ 30.,  70., 110.])
----------------------------------------------------------------------------------------------------
Vector-matrix product
equation: i,ij->j
operand 1: shape=(3,)
tensor([1., 2., 3.])
operand 2: shape=(3, 4)
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.]])
result: shape=(4,)
tensor([38., 44., 50., 56.])
----------------------------------------------------------------------------------------------------
Matrix-matrix product
equation: ik,kj->ij
operand 1: shape=(3, 4)
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.]])
operand 2: shape=(4, 2)
tensor([[1., 2.],
        [3., 4.],
        [5., 6.],
        [7., 8.]])
result: shape=(3, 2)
tensor([[ 50.,  60.],
       

tensor([[[ 110.,  120.,  130.,  140.,  150.],
         [ 246.,  272.,  298.,  324.,  350.],
         [ 382.,  424.,  466.,  508.,  550.]],

        [[1678., 1736., 1794., 1852., 1910.],
         [2134., 2208., 2282., 2356., 2430.],
         [2590., 2680., 2770., 2860., 2950.]]])

## 5. Why `->` Matters

If you omit the arrow, PyTorch keeps the labels that appear exactly once and sorts them in alphabetical order.
That can be convenient, but it can also surprise you.

For `nm,mk`, the surviving labels are `n` and `k`.
Alphabetical order is `k`, then `n`, so the implicit output is `kn`, not `nk`.

This is why explicit `->` is often the safer style in production code.


In [40]:
left = torch.arange(1.0, 7.0).reshape(2, 3)
right = torch.arange(1.0, 13.0).reshape(3, 4)

implicit = torch.einsum('nm,mk', left, right)
explicit_nk = torch.einsum('nm,mk->nk', left, right)
explicit_kn = torch.einsum('nm,mk->kn', left, right)

print('implicit shape:', implicit.shape)
print(implicit)
print('explicit -> nk shape:', explicit_nk.shape)
print(explicit_nk)
print('explicit -> kn shape:', explicit_kn.shape)
print(explicit_kn)
print('implicit equals explicit -> kn:', torch.allclose(implicit, explicit_kn))


implicit shape: torch.Size([4, 2])
tensor([[ 38.,  83.],
        [ 44.,  98.],
        [ 50., 113.],
        [ 56., 128.]])
explicit -> nk shape: torch.Size([2, 4])
tensor([[ 38.,  44.,  50.,  56.],
        [ 83.,  98., 113., 128.]])
explicit -> kn shape: torch.Size([4, 2])
tensor([[ 38.,  83.],
        [ 44.,  98.],
        [ 50., 113.],
        [ 56., 128.]])
implicit equals explicit -> kn: True


## 6. Ellipsis `...`: Hide Arbitrary Leading Dimensions

Use `...` when you care about a pattern on the last few axes, but you do not want to spell out every batch axis.

This is especially useful when the same operation should work on:

- a single batch axis
- several batch axes
- batches plus heads plus time steps

PyTorch allows ellipsis dimensions to be reduced away entirely, which is a useful feature to know.


In [41]:
T = torch.arange(1.0, 25.0).reshape(2, 3, 4)
scale = torch.tensor([1.0, 10.0, 100.0, 1000.0])
projection = torch.arange(1.0, 13.0).reshape(4, 3)
W = torch.arange(1.0, 61.0).reshape(3, 4, 5)
Z = torch.arange(1.0, 91.0).reshape(3, 5, 6)
BatchSq = torch.arange(1.0, 19.0).reshape(2, 3, 3)

describe('Sum the last axis, keep all leading axes', '...d->...', T)
describe('Dot product on the last axis for every leading index', '...d,...d->...', T, T)
describe('Scale the feature axis by a vector', '...d,d->...d', T, scale)
describe('Apply the same linear map to every leading position', '...d,df->...f', T, projection)
describe('General batched matrix multiplication', '...ij,...jk->...ik', W, Z)
describe('Batched diagonal extraction', '...ii->...i', BatchSq)
describe('PyTorch-specific: even ellipsis can be reduced away', '...d->', T)


Sum the last axis, keep all leading axes
equation: ...d->...
operand 1: shape=(2, 3, 4)
tensor([[[ 1.,  2.,  3.,  4.],
         [ 5.,  6.,  7.,  8.],
         [ 9., 10., 11., 12.]],

        [[13., 14., 15., 16.],
         [17., 18., 19., 20.],
         [21., 22., 23., 24.]]])
result: shape=(2, 3)
tensor([[10., 26., 42.],
        [58., 74., 90.]])
----------------------------------------------------------------------------------------------------
Dot product on the last axis for every leading index
equation: ...d,...d->...
operand 1: shape=(2, 3, 4)
tensor([[[ 1.,  2.,  3.,  4.],
         [ 5.,  6.,  7.,  8.],
         [ 9., 10., 11., 12.]],

        [[13., 14., 15., 16.],
         [17., 18., 19., 20.],
         [21., 22., 23., 24.]]])
operand 2: shape=(2, 3, 4)
tensor([[[ 1.,  2.,  3.,  4.],
         [ 5.,  6.,  7.,  8.],
         [ 9., 10., 11., 12.]],

        [[13., 14., 15., 16.],
         [17., 18., 19., 20.],
         [21., 22., 23., 24.]]])
result: shape=(2, 3)
tensor([[  30., 

tensor(300.)

## 7. Multi-Operand Patterns

`einsum` becomes very expressive when you use three or more operands.

These examples show how to encode pairwise similarities, quadratic forms, matrix chains, and low-rank factorizations.


In [42]:
set_a = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
set_b = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 1.0], [1.0, 1.0, 0.0]])
metric = torch.arange(1.0, 10.0).reshape(3, 3)
batch_vectors = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
M1 = torch.arange(1.0, 7.0).reshape(2, 3)
M2 = torch.arange(1.0, 13.0).reshape(3, 4)
M3 = torch.arange(1.0, 9.0).reshape(4, 2)
L = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
R = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]])

describe('Pairwise dot products between two sets of vectors', 'nd,md->nm', set_a, set_b)
describe('One quadratic form per batch item', 'bi,ij,bj->b', batch_vectors, metric, batch_vectors)
describe('Three-matrix chain product', 'ab,bc,cd->ad', M1, M2, M3)
describe('Low-rank reconstruction', 'ir,rj->ij', L, R)


Pairwise dot products between two sets of vectors
equation: nd,md->nm
operand 1: shape=(2, 3)
tensor([[1., 2., 3.],
        [4., 5., 6.]])
operand 2: shape=(3, 3)
tensor([[1., 0., 1.],
        [0., 1., 1.],
        [1., 1., 0.]])
result: shape=(2, 3)
tensor([[ 4.,  5.,  3.],
        [10., 11.,  9.]])
----------------------------------------------------------------------------------------------------
One quadratic form per batch item
equation: bi,ij,bj->b
operand 1: shape=(2, 3)
tensor([[1., 2., 3.],
        [4., 5., 6.]])
operand 2: shape=(3, 3)
tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])
operand 3: shape=(2, 3)
tensor([[1., 2., 3.],
        [4., 5., 6.]])
result: shape=(2,)
tensor([ 228., 1245.])
----------------------------------------------------------------------------------------------------
Three-matrix chain product
equation: ab,bc,cd->ad
operand 1: shape=(2, 3)
tensor([[1., 2., 3.],
        [4., 5., 6.]])
operand 2: shape=(3, 4)
tensor([[ 1.,  2.,  3.,  4

tensor([[ 1.,  2.,  3.],
        [ 3.,  4.,  7.],
        [ 5.,  6., 11.]])

## 8. A Real Deep-Learning Example: Attention

Here is a common pattern from transformers.

Legend:

- `b`: batch
- `h`: head
- `q`: query position
- `k`: key position
- `d`: feature dimension used for matching queries and keys
- `v`: value dimension

Two common equations are:

- `bhqd,bhkd->bhqk` for attention scores
- `bhqk,bhkv->bhqv` for the weighted sum of values

Even if you do not memorize the exact letters, the pattern becomes readable once you track which axes disappear and which axes survive.


In [43]:
Q = torch.arange(1.0, 17.0).reshape(1, 2, 2, 4)
K = torch.arange(1.0, 25.0).reshape(1, 2, 3, 4)
V = torch.arange(1.0, 13.0).reshape(1, 2, 3, 2)

scores = describe('Attention scores', 'bhqd,bhkd->bhqk', Q, K)
weights = torch.softmax(scores / (Q.size(-1) ** 0.5), dim=-1)
print('Softmax-normalized weights:')
print(weights)
print('-' * 100)
context = describe('Attention context', 'bhqk,bhkv->bhqv', weights, V)


Attention scores
equation: bhqd,bhkd->bhqk
operand 1: shape=(1, 2, 2, 4)
tensor([[[[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.]],

         [[ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]]])
operand 2: shape=(1, 2, 3, 4)
tensor([[[[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.]],

         [[13., 14., 15., 16.],
          [17., 18., 19., 20.],
          [21., 22., 23., 24.]]]])
result: shape=(1, 2, 2, 3)
tensor([[[[  30.,   70.,  110.],
          [  70.,  174.,  278.]],

         [[ 614.,  782.,  950.],
          [ 846., 1078., 1310.]]]])
----------------------------------------------------------------------------------------------------
Softmax-normalized weights:
tensor([[[[0.000, 0.000, 1.000],
          [0.000, 0.000, 1.000]],

         [[0.000, 0.000, 1.000],
          [0.000, 0.000, 1.000]]]])
----------------------------------------------------------------------------------------------------
Attention context
equation: bhqk,bhk

## 9. Common Mistakes and Debugging Rules

When an `einsum` expression fails, the bug is usually one of these:

- the number of labels does not match the tensor rank
- a repeated label inside one operand is not actually square
- an output label was never introduced by any input
- the intended output order was not written explicitly with `->`
- an ellipsis hides batch axes that do not broadcast together

A strong workflow is:

1. start with very small tensors
2. write a legend for every label
3. compare against a familiar PyTorch operation such as `matmul`, `dot`, `outer`, `sum`, or `diagonal`
4. add `assert torch.allclose(...)` while you are learning or refactoring


In [44]:
# A few quick sanity checks against familiar PyTorch operations
assert torch.allclose(torch.einsum('i,i->', v, w), torch.dot(v, w))
assert torch.allclose(torch.einsum('i,j->ij', v, w), torch.outer(v, w))
assert torch.allclose(torch.einsum('ii->i', A), A.diagonal())
assert torch.allclose(torch.einsum('ik,kj->ij', B, C), B @ C)
assert torch.allclose(torch.einsum('bij,bjk->bik', X, Y), torch.matmul(X, Y))
assert torch.allclose(torch.einsum('...d->...', T), T.sum(dim=-1))
assert torch.allclose(torch.einsum('...ij,...jk->...ik', W, Z), torch.matmul(W, Z))

print('All sanity checks passed.')


All sanity checks passed.


## 10. Compact Cheat Sheet

| Pattern | Meaning |
| --- | --- |
| `i->i` | identity on a vector |
| `ij->ji` | transpose |
| `ij->i` | sum over columns |
| `ij->j` | sum over rows |
| `ij->` | sum every element |
| `ii->i` | diagonal extraction |
| `ii->` | trace |
| `i,i->` | dot product |
| `i,j->ij` | outer product |
| `ij,ij->ij` | elementwise product |
| `ij,ij->` | Frobenius inner product |
| `ij,j->i` | matrix-vector product |
| `i,ij->j` | vector-matrix product |
| `ik,kj->ij` | matrix multiplication |
| `bij,bjk->bik` | batched matrix multiplication |
| `nd,md->nm` | pairwise dot products between two sets of vectors |
| `bi,ij,bj->b` | batch of quadratic forms |
| `...d->...` | sum the last axis while keeping all leading axes |
| `...d,...d->...` | dot product over the last axis for arbitrary leading dims |
| `...ij,...jk->...ik` | general batched matrix multiplication |
| `bhqd,bhkd->bhqk` | attention scores |
| `bhqk,bhkv->bhqv` | attention-weighted values |

If you can read the examples above without translating them back to English every time, you already understand most real-world `torch.einsum` usage.
